# MuscleMap WB + MedSAM Refinement — Water (Lambda)

Uses existing MuscleMap whole-body segmentations as bounding-box prompts for MedSAM.
For each **water** image the corresponding `_dseg.nii.gz` is loaded; per-muscle
bounding boxes are extracted slice-by-slice and fed to MedSAM.

MedSAM embedding is computed **once per slice** and reused for all muscles.
MedSAM runs on CPU to leave GPU memory for other work.

## Before running — upload to Lambda

```bash
# water images
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/myosegmenTUM \
  ubuntu@129.80.59.179:~/

# MuscleMap WB water segmentations (produced by musclemap_water_lambda.ipynb)
# These should already be on Lambda in ~/musclemap_water_segs/ if that notebook was run there.

# MedSAM checkpoint
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  "/tmp/docker-desktop-root/run/desktop/mnt/host/c/Users/docto/AppData/Local/Dafne-imaging/Dafne/models/medsam_vit_b.pth" \
  ubuntu@129.80.59.179:~/medsam_vit_b.pth
```

## Download results when done

```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@129.80.59.179:~/MuscleMap_WB_medsam_water/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/MuscleMap_WB_medsam_water/
```

**Terminate the instance when done.**

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'dafne-dl', 'SimpleITK', 'scikit-image'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/facebookresearch/segment-anything.git'])

In [ ]:
import glob
import os
import numpy as np
import SimpleITK as sitk
import torch
import torch.nn.functional as F
from skimage import transform
from segment_anything import sam_model_registry

In [ ]:
# inlined utilities (replaces dafne.utils.sam_mask_refine)
def enlarge_bounding_box(mask, margin=5):
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    H, W = mask.shape
    return np.array([
        max(0, cmin - margin), max(0, rmin - margin),
        min(W - 1, cmax + margin), min(H - 1, rmax + margin),
    ], dtype=float)

def medsam_inference(medsam_model, img_embed, box_1024, H, W):
    box_torch = torch.as_tensor(box_1024, dtype=torch.float, device=img_embed.device)
    if box_torch.ndim == 2:
        box_torch = box_torch[:, None, :]
    sparse_embeddings, dense_embeddings = medsam_model.prompt_encoder(
        points=None, boxes=box_torch, masks=None
    )
    low_res_logits, _ = medsam_model.mask_decoder(
        image_embeddings=img_embed,
        image_pe=medsam_model.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse_embeddings,
        dense_prompt_embeddings=dense_embeddings,
        multimask_output=False,
    )
    low_res_pred = F.interpolate(
        torch.sigmoid(low_res_logits), size=(H, W),
        mode='bilinear', align_corners=False
    )
    return (low_res_pred.squeeze().detach().cpu().numpy() > 0.5).astype(np.uint8)

In [ ]:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
SAM_DEVICE = 'cpu'   # MedSAM encoder is memory-heavy; keep on CPU

MM_SEGS_DIR = os.path.expanduser('~/musclemap_water_segs')
IMAGE_GLOB  = os.path.expanduser('~/myosegmenTUM/*/ImageData/*_WATER/*_WATER_stack*.nii')
OUTPUT_DIR  = os.path.expanduser('~/MuscleMap_WB_medsam_water')
SAM_CKPT    = os.path.expanduser('~/medsam_vit_b.pth')

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(SAM_CKPT):
    raise FileNotFoundError(f'MedSAM checkpoint not found: {SAM_CKPT} — upload it first.')

print('Device    :', DEVICE)
print('SAM device:', SAM_DEVICE)
print('MM segs   :', MM_SEGS_DIR)
print('Output    :', OUTPUT_DIR)

In [ ]:
# MuscleMap WB label map
LABEL_MAP = {
    7101: 'Vastus_Lateralis_L',
    7102: 'Vastus_Lateralis_R',
    7111: 'Vastus_Intermedius_L',
    7112: 'Vastus_Intermedius_R',
    7121: 'Vastus_Medialis_L',
    7122: 'Vastus_Medialis_R',
    7131: 'Rectus_Femoris_L',
    7132: 'Rectus_Femoris_R',
    7141: 'Sartorius_L',
    7142: 'Sartorius_R',
    7151: 'Gracilis_L',
    7152: 'Gracilis_R',
    7201: 'Adductor_Magnus_L',
    7202: 'Adductor_Magnus_R',
    7161: 'Semimembranosus_L',
    7162: 'Semimembranosus_R',
    7171: 'Semitendinosus_L',
    7172: 'Semitendinosus_R',
    7181: 'Biceps_Femoris_L',
    7182: 'Biceps_Femoris_R',
}
print(f'{len(LABEL_MAP)} muscle labels defined')

In [ ]:
# load MedSAM once on CPU
sam_model = sam_model_registry['vit_b'](checkpoint=SAM_CKPT)
sam_model.to(device=SAM_DEVICE)
sam_model.eval()
print('MedSAM loaded on', SAM_DEVICE)

In [ ]:
image_files = sorted(glob.glob(IMAGE_GLOB))
print(f'Found {len(image_files)} water images')

matched, missing = [], []
for nii_path in image_files:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    seg_path = os.path.join(MM_SEGS_DIR, f'{stem}_dseg.nii.gz')
    if os.path.exists(seg_path):
        matched.append(nii_path)
    else:
        missing.append(stem)

print(f'  {len(matched)} have a MuscleMap seg, {len(missing)} do not')
if missing:
    print('  Missing segs for:', missing[:5], '...' if len(missing) > 5 else '')

In [ ]:
for nii_path in matched:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    out_path = os.path.join(OUTPUT_DIR, f'{stem}_mm_medsam.npz')

    if os.path.exists(out_path):
        print(f'Skipping (already done): {stem}')
        continue

    seg_path  = os.path.join(MM_SEGS_DIR, f'{stem}_dseg.nii.gz')
    print(f'\nProcessing: {nii_path}')

    img_sitk  = sitk.ReadImage(nii_path)
    img_array = sitk.GetArrayFromImage(img_sitk).astype(float)
    H, W      = img_array.shape[1], img_array.shape[2]
    print(f'  Image shape: {img_array.shape}')

    seg_sitk  = sitk.ReadImage(seg_path)
    seg_array = sitk.GetArrayFromImage(seg_sitk)
    print(f'  Seg shape  : {seg_array.shape}')

    all_masks = {}

    for slice_idx in range(img_array.shape[0]):
        slice_2d  = img_array[slice_idx]
        seg_slice = seg_array[slice_idx]

        # MedSAM embedding — once per slice, reused for all muscles
        img_norm   = slice_2d * 255.0 / (slice_2d.max() + 1e-8)
        img_3c     = np.repeat(img_norm[:, :, None], 3, axis=-1)
        img_1024   = transform.resize(
            img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True
        ).astype(np.uint8)
        img_1024   = (img_1024 - img_1024.min()) / np.clip(
            img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None
        )
        img_tensor = torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(SAM_DEVICE)
        with torch.no_grad():
            image_embedding = sam_model.image_encoder(img_tensor)
        del img_tensor

        for label_idx, muscle_name in LABEL_MAP.items():
            mask_arr = (seg_slice == label_idx).astype(np.uint8)

            if mask_arr.any():
                bbox     = enlarge_bounding_box(mask_arr)
                box_1024 = bbox / np.array([W, H, W, H]) * 1024
                box_1024 = box_1024[None, None, :]
                refined  = medsam_inference(sam_model, image_embedding, box_1024, H, W)
            else:
                refined = mask_arr

            if muscle_name not in all_masks:
                all_masks[muscle_name] = np.zeros(img_array.shape, dtype=np.uint8)
            all_masks[muscle_name][slice_idx] = refined.astype(np.uint8)

        del image_embedding

        if (slice_idx + 1) % 5 == 0 or slice_idx == img_array.shape[0] - 1:
            print(f'  slice {slice_idx + 1}/{img_array.shape[0]} done')

    np.savez_compressed(out_path, **all_masks)
    print(f'  Saved → {out_path}')

print('\nAll done.')

In [ ]:
# sanity check — reload one result
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.npz')))
print(f'Total output files: {len(results)}')
if results:
    sample = np.load(results[0])
    print('Sample file:', results[0])
    for name in sample.files:
        arr = sample[name]
        print(f'  {name}: shape={arr.shape}  positive voxels={arr.sum()}')